# ClinTrial-LM — try the fine-tuned model

Loads the QLoRA-fine-tuned **Qwen2.5-7B** clinical-trial assistant and runs all four tasks it was trained on:

1. **Eligibility extraction** — free-text criteria → structured JSON
2. **Plain-language summary** — trial description → patient-friendly summary
3. **Condition Q&A** — what does this trial study?
4. **Phase classification** — what phase is it?

Repo: https://github.com/omkar-droid/clintrial-lm  ·  Needs a GPU (~16 GB in 4-bit).

> ⚕️ Research demo — **not medical advice**.

In [ ]:
# !pip install -q transformers peft accelerate bitsandbytes

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Point this at the merged model on the Hub, or a local path.
MODEL_ID = "omkar-droid/clintrial-qwen2.5-7b-sft"   # or: "merged/qlora-sft"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
model.eval()
print("loaded", MODEL_ID)

In [ ]:
SYSTEM = (
    "You are a clinical research assistant. You help patients and clinicians understand "
    "clinical trials. Answer only from the information provided, be precise, and never "
    "invent eligibility criteria, conditions, or outcomes."
)

@torch.no_grad()
def ask(user_prompt, max_new_tokens=1024):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_prompt}]
    enc = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

## 1. Eligibility extraction → JSON
The headline task: messy registry text in, machine-readable criteria out.

In [ ]:
criteria = """Inclusion Criteria:
1. Adults aged 18 years or older with a confirmed diagnosis of type 2 diabetes
2. HbA1c between 7.0% and 10.5% at screening
3. Able to provide written informed consent

Exclusion Criteria:
1. History of severe hypoglycaemia within the last 6 months
2. Estimated glomerular filtration rate below 30 mL/min
3. Pregnancy or breastfeeding
"""

prompt = (
    'Extract the eligibility criteria from the trial text below into JSON with two '
    'lists, "inclusion" and "exclusion". Copy each criterion verbatim; do not add any.\n\n'
    f"Trial eligibility text:\n{criteria}"
)
print(ask(prompt))

## 2. Plain-language summary

In [ ]:
prompt = (
    "Write a short, plain-language summary (3-5 sentences) of this clinical trial for a "
    "patient audience.\n\n"
    "Title: A Randomized Phase 3 Study of Semaglutide Versus Placebo for Weight Management "
    "in Adults With Obesity\n"
    "Conditions: Obesity\n"
    "Interventions: Semaglutide, Placebo"
)
print(ask(prompt))

## 3 & 4. Condition Q&A and phase classification
These are the tasks where fine-tuning mattered most — the base model knew the answers but
wouldn't produce them in the required format (phase exact-match went 0.00 → 0.79).

In [ ]:
summary = (
    "This phase 3 clinical trial evaluates whether semaglutide reduces body weight in adults "
    "with obesity. Participants are randomly assigned to semaglutide or placebo and followed "
    "for 68 weeks to measure change in body weight and safety outcomes."
)

print("CONDITIONS:", ask(f"Based on this trial summary, what medical conditions is it studying? List them.\n\n{summary}", 128))
print("PHASE     :", ask(f"What clinical trial phase is described below? Answer with the phase(s) only.\n\n{summary}", 32))